# 03. PFN 구현 디테일

이 노트북은 [`02_pfn_paper_walkthrough.ipynb`](02_pfn_paper_walkthrough.ipynb)(세션 1, 이론)에서 이어집니다. 02에서 PFN이 무엇이고(1절), task/데이터셋을 어떻게 구성하며(2절), NLL 손실이 왜 PPD로 수렴하고(3~4절), prior-fitting 루프가 어떻게 도는지(5절)까지 다뤘습니다.

여기서는 그 이론을 **실제로 구현할 때의 선택**과 **세션 2 재현 준비**를 다룹니다. 절 번호는 논문 흐름과 02 노트북에 이어지도록 6절부터 매깁니다 — 본문에서 "2절", "4절" 등은 02 노트북을 가리킵니다.

6. 회귀 출력 — Riemann / bar distribution: 예측분포를 어떤 형태로 낼 것인가
7. 아키텍처 — set을 처리하는 Transformer encoder
8. GP prior 집중 — 세션 2 재현 대상의 "정답"(analytic posterior)
9. 요약과 세션 2로의 다리

아래 import 블록은 02와 동일합니다 (이 노트북을 단독으로 실행할 수 있도록 다시 포함).

## `pfn_cookbook` 불러오기

chapter 01과 동일하게, 로컬에서는 설치된 패키지를, Databricks처럼 미설치 환경에서는 `src/`를 import 경로에 추가합니다.

In [ ]:
try:
    import pfn_cookbook
except ImportError:
    import sys
    from pathlib import Path

    sys.path.insert(0, str(Path.cwd().parent / "src"))
    import pfn_cookbook

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

from pfn_cookbook import set_seed

set_seed(0)  # 재현성을 위해 난수 시드를 고정한다.

## 6. 회귀 출력 — Riemann / bar distribution

분류는 라벨이 이산이라 softmax로 예측분포를 내면 됩니다. 그런데 **회귀**는 $y$가 연속이고, 게다가 PPD가 **비대칭·다봉(multimodal)** 일 수 있습니다 (여러 그럴듯한 함수가 공존할 때).  
단일 가우시안 출력으로는 이걸 담지 못합니다.

논문의 해법은 **출력 구간을 여러 bucket(막대)으로 이산화**하는 것입니다 — *Riemann distribution* 또는 *bar distribution*이라 부릅니다.

- 출력 범위를 경계 $b_0 < b_1 < \dots < b_B$로 잘라 $B$개 bucket을 만든다
- 신경망은 bucket에 대한 **범주분포**(softmax)를 낸다
- 각 bucket 안에서는 밀도가 균일하다고 보면, 예측 밀도는 **계단형(piecewise-constant)** 함수

$$
q(y) = \frac{p_k}{b_k - b_{k-1}}, \qquad y \in [b_{k-1}, b_k), \qquad \sum_k p_k = 1
$$

학습 시 NLL은 $y$가 든 bucket의 밀도에 로그를 취한 값입니다: $-\log q(y)$. bucket이 충분히 촘촘하면 **임의 형태의 예측분포**(다봉 포함)를 근사할 수 있습니다. 아래에서 이봉 분포를 bar distribution으로 표현해봅니다.

In [ ]:
# 목표: 이봉(bimodal) 예측분포 — 단일 가우시안으로는 못 담는 형태
def target_pdf(y):
    return 0.6 * norm.pdf(y, -1.5, 0.4) + 0.4 * norm.pdf(y, 1.8, 0.6)

def target_cdf(y):
    return 0.6 * norm.cdf(y, -1.5, 0.4) + 0.4 * norm.cdf(y, 1.8, 0.6)

B = 40
edges = np.linspace(-4, 4, B + 1)          # bucket 경계
widths = np.diff(edges)                      # bucket 폭
probs = target_cdf(edges[1:]) - target_cdf(edges[:-1])  # 각 bucket 확률 p_k
probs /= probs.sum()                         # 범주분포로 정규화 (softmax가 낼 값에 해당)
density = probs / widths                      # 계단형 밀도 = p_k / 폭

# 샘플 y 하나의 NLL 계산 예시
y_obs = 1.9
k = np.searchsorted(edges, y_obs) - 1        # y가 든 bucket 인덱스
nll = -np.log(density[k])
print(f"y={y_obs} 는 bucket {k} 에 속하고, bar-dist NLL = -log q(y) = {nll:.3f}")

fig, ax = plt.subplots(figsize=(7.5, 4.2))
grid = np.linspace(-4, 4, 400)
ax.plot(grid, target_pdf(grid), color="C3", lw=2, label="target PPD (bimodal)")
ax.stairs(density, edges, color="C0", fill=True, alpha=0.35, label=f"bar distribution (B={B})")
ax.set_xlabel("y")
ax.set_ylabel("density")
ax.set_title("A bar distribution approximates a multimodal predictive density")
ax.legend()
plt.show()

## 7. 아키텍처 — set을 처리하는 Transformer *encoder*

PFN은 GPT류의 **decoder**(인과 마스킹, 다음 토큰 예측)가 아니라 **encoder** 를 사용합니다. 입력은 *순서가 없는 집합*이기 때문입니다.

- **토큰화**: 각 학습 예제 $(x_i, y_i)$가 토큰 하나. 쿼리 지점은 $x$만 있고 $y$는 가려진 토큰.
- **attention 마스킹**:
  - 학습 토큰들은 서로 자유롭게 attend (context를 종합)
  - 쿼리 토큰은 **학습 토큰에만** attend하고, **다른 쿼리 토큰에는 attend하지 않음** — 그래야 한 배치에 여러 쿼리를 넣어도 서로의 답이 새지 않음
  - 인과(causal) 마스킹이 없으므로 학습 예제의 **순서에 불변**(permutation invariant)
- **출력 head**: 각 쿼리 위치에서 예측분포를 낸다 — 분류면 softmax, 회귀면 6절의 bar distribution.

![PFN Figure 1(a)](images/pfn_fig1a.png)

이 attention 흐름은 2절 Figure 1(a)의 화살표에 그대로 나타나 있습니다 — 파란 화살표는 context 토큰끼리, 빨간 화살표는 각 query가 context에만 attend.

![PFN encoder: set-to-prediction 흐름 (tokens → encoder → 예측분포)](images/pfn_encoder_arch.svg)

**"데이터셋 전체를 문맥으로 읽고, 쿼리에 대한 예측분포를 내는 set-to-prediction Transformer."** 가중치 업데이트 없이 문맥만으로 새 데이터셋에 적응하는 in-context learning이 여기서 나옵니다.

## 8. GP prior

이 다음에는 **prior = 가우시안 프로세스(GP)** 로 PFN을 학습해, GP 회귀를 재현합니다.  
GP를 고른 이유는 **posterior가 닫힌형으로 정확히 계산**되기 때문입니다 — 즉 "PFN이 논문만큼 잘 하는가"를 **정답과 직접 대조**할 수 있습니다.

GP는 *함수 위의 prior*입니다. 커널 $k(x, x')$가 함수의 매끄러움을 정합니다 (여기선 RBF).  
관측 $D = (X_{tr}, y_{tr})$ (관측 노이즈 $\sigma^2$)가 주어지면, 임의의 쿼리 $X_*$에서 posterior는 다시 가우시안이고 평균·공분산이 닫힌형입니다.

$$
\mu_* = K_{*,tr}\,(K_{tr,tr} + \sigma^2 I)^{-1} y_{tr}, \qquad
\Sigma_* = K_{*,*} - K_{*,tr}\,(K_{tr,tr} + \sigma^2 I)^{-1} K_{tr,*}
$$

**2절의 task/데이터셋 구성을 GP에 적용하면 (논문 §5.1, 부록 F.1)**: 입력 $x_i$를 단위 큐브에서 균일하게 뽑고, task(함수)는 zero-mean GP에서 $y \sim \mathcal{N}(0, K)$로 **한 번에** 샘플링 합니다.  
($K_{ij} = k(x_i, x_j)$). 실험 커널은 length scale $0.6$짜리 RBF에 대각 노이즈 $10^{-4}$입니다.

아래에서 (왼쪽) GP prior에서 뽑은 함수 샘플 — 이게 PFN 학습 데이터의 원천 — 과 (오른쪽) 관측이 주어졌을 때의 **analytic posterior**를 그립니다.  
**세션 2에서 학습한 PFN의 예측분포는 오른쪽 그림에 겹쳐야** 재현에 성공한다고 판단합니다.

In [ ]:
def rbf_kernel(a, b, lengthscale=1.0, variance=1.0):
    d = a[:, None] - b[None, :]
    return variance * np.exp(-0.5 * (d / lengthscale) ** 2)

X = np.linspace(-5, 5, 200)
K = rbf_kernel(X, X) + 1e-9 * np.eye(len(X))
L = np.linalg.cholesky(K)

# ── 관측(하나의 task에서 나온 데이터셋) ──
X_tr = np.array([-4.0, -2.5, -0.5, 1.0, 3.5])
y_tr = np.sin(X_tr) + 0.1 * np.random.randn(X_tr.size)
noise = 0.1

# ── analytic GP posterior ──
K_tr = rbf_kernel(X_tr, X_tr) + noise**2 * np.eye(X_tr.size)
K_s = rbf_kernel(X, X_tr)
K_inv = np.linalg.inv(K_tr)
mu = K_s @ K_inv @ y_tr
cov = rbf_kernel(X, X) - K_s @ K_inv @ K_s.T
sd = np.sqrt(np.clip(np.diag(cov), 0, None))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))

ax = axes[0]
for _ in range(6):
    f = L @ np.random.randn(len(X))  # GP prior에서 함수 샘플
    ax.plot(X, f, lw=1.0, alpha=0.8)
ax.set_title("GP prior function samples (source of PFN training data)")
ax.set_xlabel("x"); ax.set_ylabel("f(x)")

ax = axes[1]
ax.fill_between(X, mu - 2 * sd, mu + 2 * sd, color="C0", alpha=0.2, label="±2σ")
ax.plot(X, mu, color="C0", lw=2, label="posterior mean")
ax.scatter(X_tr, y_tr, c="black", s=35, zorder=5, label="observations D")
ax.set_title("analytic GP posterior (PFN target for session 2)")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend(loc="upper left", fontsize=8)
plt.show()